In [27]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import base64
import json
from Crypto.Protocol.KDF import PBKDF2
from Crypto.Hash import SHA256
import gradio as gr
import contextlib
import io
import re
import os

In [28]:
def derivar_clave(passphrase, salt):
    """
    Deriva una clave AES utilizando PBKDF2 con HMAC-SHA256.

    :param passphrase: Frase de contraseña.
    :param salt: Sal (bytes).
    :return: Clave derivada (bytes).
    """
    return PBKDF2(passphrase, salt, dkLen=32, count=100000, hmac_hash_module=SHA256)


In [29]:
def codificar_mensaje(mensaje, passphrase_, premium=None, dias=0, dfa=123456, usuario_inicial=0):
    """
    Cifra un mensaje estructurado en JSON utilizando AES en modo CFB.

    :param mensaje: Mensaje a cifrar.
    :param passphrase_: Frase de contraseña.
    :param premium: Tipo de premium.
    :param dias: Número de días.
    :param dfa: Valor DFA.
    :param usuario_inicial: Usuario inicial.
    :return: Mensaje cifrado en Base64.
    """
    try:
        # Sal fija utilizada para derivar la clave
        salt = os.environ['SALT']
        clave_bytes = derivar_clave(passphrase_, salt)
        iv = get_random_bytes(16)  # IV aleatorio
        cifrador = AES.new(clave_bytes, AES.MODE_CFB, iv)

        # Simulación de funciones externas (reemplaza con tus implementaciones reales)
        bloque_btc = obtener_ultimo_bloque_confirmado()
        dias_a_bloques = calcular_bloques_en_dias(int(dias))
        desbloqueo = int(bloque_btc) + int(dias_a_bloques)

        # Determinar la cabecera según el valor de 'premium'
        if premium == 'hombre_muerto':
            cabecera = "bloq. por hombre muerto"
        elif premium in ['bloqueo', 'bloqueo_free']:
            cabecera = "bloq. por bloqueo"
        elif premium == 'hombre_muerto_free':
            desbloqueo = int(bloque_btc) + int(2)
            cabecera = "bloq. por hombre muerto"
        elif premium == 'codificacion':
            cabecera = "bloq. por codificacion"
        else:
            cabecera = "vacio"

        # Estructurar los datos en un diccionario
        datos = {
            "bloque_origen": bloque_btc,
            "bloque_desbloqueo": desbloqueo,
            "mensaje": mensaje,
            "cabecera": cabecera,
            "dfa": dfa,
            "usuario_inicial": usuario_inicial
        }

        # Convertir el diccionario a una cadena JSON
        mensaje_json = json.dumps(datos)
        mensaje_json_bytes = mensaje_json.encode('utf-8')

        # Cifrar el mensaje JSON
        mensaje_cifrado = iv + cifrador.encrypt(mensaje_json_bytes)
        mensaje_cifrado_base64 = base64.urlsafe_b64encode(mensaje_cifrado).decode('utf-8')

        return mensaje_cifrado_base64
    except Exception as e:
        return f"Error: {str(e)}"


In [30]:
def descodificar_mensaje(mensaje_cifrado_base64, passphrase_, salt=os.environ['SALT']):
    """
    Descifra un mensaje cifrado en Base64 utilizando AES en modo CFB y extrae el campo 'mensaje'.

    :param mensaje_cifrado_base64: Mensaje cifrado en Base64.
    :param passphrase_: Frase de contraseña.
    :param salt: Sal utilizada para derivar la clave (bytes).
    :return: Campo 'mensaje' descifrado o mensaje de error.
    """
    try:
        # Derivar la clave usando la passphrase y la sal
        clave_bytes = derivar_clave(passphrase_, salt)

        # Decodificar el mensaje cifrado de Base64
        mensaje_cifrado = base64.urlsafe_b64decode(mensaje_cifrado_base64)

        # Verificar que el mensaje cifrado tenga al menos 16 bytes para el IV
        if len(mensaje_cifrado) < 16:
            raise ValueError("El mensaje cifrado es demasiado corto para contener un IV.")

        # Extraer el IV (primeros 16 bytes)
        iv = mensaje_cifrado[:16]
        mensaje_cifrado = mensaje_cifrado[16:]

        # Inicializar el descifrador
        descifrador = AES.new(clave_bytes, AES.MODE_CFB, iv)

        # Descifrar el mensaje
        mensaje_descifrado_bytes = descifrador.decrypt(mensaje_cifrado)

        # Decodificar los bytes descifrados a cadena UTF-8
        mensaje_descifrado = mensaje_descifrado_bytes.decode('utf-8')

        # Convertir la cadena JSON de vuelta a un diccionario
        datos = json.loads(mensaje_descifrado)

        # Extraer el campo 'mensaje'
        mensaje = datos.get("mensaje", "")

        return mensaje

    except Exception as e:
        return f"Error de descifrado: {str(e)}"


In [31]:
def obtener_ultimo_bloque_confirmado():
    """
    Simula la obtención del último bloque confirmado.
    """
    return "123456"

def calcular_bloques_en_dias(dias):
    """
    Calcula el número de bloques en función de los días.

    :param dias: Número de días.
    :return: Número de bloques.
    """
    return dias * 144  # Supongamos 144 bloques por día


In [32]:
# Definir la interfaz de Gradio
interface = gr.Interface(
    fn=descodificar_mensaje,  # Función que se ejecutará
    inputs=[
        gr.Textbox(lines=2, placeholder="Introduce el mensaje cifrado en Base64 aquí...", label="Mensaje Cifrado (Base64)"),
        gr.Textbox(lines=1, placeholder="Introduce la contraseña aquí...", label="Contraseña")
    ],
    outputs=gr.Textbox(label="Mensaje Descifrado"),
    description="Introduce el mensaje cifrado en Base64 y la contraseña para descifrarlo. Solo se mostrará el campo 'mensaje'."
)


In [33]:
def launch_gradio_interface(interface):
    # Crear un buffer para capturar la salida
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer), contextlib.redirect_stderr(buffer):
        # Lanzar Gradio con share=True para obtener una URL pública
        server = interface.launch(share=True, inline=False)
    # Extraer la URL de la salida capturada usando regex
    output = buffer.getvalue()
    match = re.search(r"Running on public URL: (https?://[^\s]+)", output)
    if match:
        share_url = match.group(1)
    else:
        share_url = "No se pudo obtener la URL."
    return share_url


In [34]:
# Lanzar la interfaz y capturar la URL
share_url = launch_gradio_interface(interface)

# Mostrar la URL de manera limpia
print(f"Interfaz disponible en: {share_url}")


Interfaz disponible en: https://2773bf71663d34e39d.gradio.live


In [35]:
def test_cifrado_descifrado():
    mensaje_original = "Este es un mensaje de prueba."
    passphrase = "mi_contraseña_segura"
    premium = "bloqueo"
    dias = 10
    dfa = 123456
    usuario_inicial = 7890

    # Cifrar el mensaje
    mensaje_cifrado = codificar_mensaje(mensaje_original, passphrase, premium, dias, dfa, usuario_inicial)
    print(f"Mensaje Cifrado (Base64): {mensaje_cifrado}")

    # Descifrar el mensaje
    mensaje_descifrado = descodificar_mensaje(mensaje_cifrado, passphrase)
    {mensaje_descifrado}


In [36]:
descodificar_mensaje('Wnnqsj8jdbQnhlv0_IfGk5N2f4mx0Xv0i1wanQt3INbR_A_cfR0KBSPlZhFIOf2ltuigqZSygenUUYxNzuUc7Zgy76id8vTQSyhNqG6-ks7ejLPMu6hKXv32hMoRUwc4VKb2aLs08XnTmMrP7omoSqCOykJ45Mm6FVwdokdFz7jVCtyYkeMufAiOwBMEAxaVtichiWeCTTt_xek67yJK3kfxb-681Kb7bVTVraCuk46jlzqnjxONyDcNSJTv7tjZ1VvVxPaKeGXJVk6Xoi4Dct-FMw7FYkcNuFVuc19SYpWjcbodrKJOQr2wur_MIF_TmKqF5r-53vR-TRyN53tuSNWTr6Cp15sgoVuCSAqxQLCg_Nwb8qGW8ATsasv-DRqSFBEyph7uJceI0w1I12pKm1dW8ojte6CN7CYzslt75cqRPBmxeGWG8mZu93vm_GvFzC38nU-43Yb6ZzQxPnA65IeFk7C-kEf54bbcrI5-nsjbRX_SgD9Yf3nS2bcbTTQGuJfu9tpCrH2QQhBTCo3lwfCTmQUCEfs5XRpYk-yCaK-qAZAcznd4rhtYEA0HGu1vM9sIyyVr3YAQ1LrH9aa6qH75mhsogsPYgID_kKcFWWFvJm1oz5_X7Pj7egF2kxlWjkThqCo=', '4218039')

"Error de descifrado: 'utf-8' codec can't decode byte 0x90 in position 0: invalid start byte"

In [7]:
!pip install fastapi uvicorn nest_asyncio pyngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 4.5 MB/s eta 0:00:00


In [8]:
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, HTTPException
import hmac
import hashlib
import time
import secrets
from typing import Dict

# Instancia de FastAPI
app = FastAPI()

# Nuestro "secreto real" que no queremos exponer
SUPER_SECRET_KEY = b"clave_interna_que_solo_el_servidor_conoce"

TOKEN_VALIDITY_SECONDS = 60  # Validez del token en segundos
active_tokens: Dict[str, float] = {}  # Diccionario para tokens activos (opcional)

def generar_token_efimero():
    """Genera un token único, firmado con HMAC, que caduca en TOKEN_VALIDITY_SECONDS."""
    nonce = secrets.token_hex(8)
    timestamp = str(int(time.time()))
    mensaje = f"{timestamp}:{nonce}"
    firma = hmac.new(SUPER_SECRET_KEY, mensaje.encode(), hashlib.sha256).hexdigest()
    token = f"{mensaje}:{firma}"
    return token

def validar_token(token: str):
    """Valida la firma y la vigencia del token."""
    partes = token.split(":")
    if len(partes) != 3:
        return False, "Token mal formado"

    timestamp_str, nonce, firma_recibida = partes
    try:
        timestamp = int(timestamp_str)
    except:
        return False, "Timestamp no válido"

    # Verificar caducidad
    if time.time() - timestamp > TOKEN_VALIDITY_SECONDS:
        return False, "Token caducado"

    # Recalcular firma esperada
    mensaje = f"{timestamp_str}:{nonce}"
    firma_esperada = hmac.new(SUPER_SECRET_KEY, mensaje.encode(), hashlib.sha256).hexdigest()

    if not hmac.compare_digest(firma_esperada, firma_recibida):
        return False, "Firma inválida"

    return True, None

@app.get("/generate-token")
def generate_token():
    """
    Genera un token efímero y lo guarda opcionalmente en 'active_tokens'.
    """
    token = generar_token_efimero()
    active_tokens[token] = time.time()
    return {"token": token, "expires_in_seconds": TOKEN_VALIDITY_SECONDS}

@app.get("/use-token")
def use_token(token: str):
    """
    Endpoint que valida y "usa" el token.
    En la realidad, aquí podrías hacer una acción que requiere 'SUPER_SECRET_KEY'.
    """
    valido, error_msg = validar_token(token)
    if not valido:
        raise HTTPException(status_code=403, detail=error_msg)

    # Si quieres token de un solo uso:
    if token in active_tokens:
        del active_tokens[token]

    # Aquí harías algo que requiera el secreto real...
    # Por ejemplo, invocar un servicio externo, generar una firma, etc.

    return {"message": "Token válido. Se ha usado el secreto internamente."}


In [9]:
import uvicorn
from pyngrok import ngrok

# Arrancar uvicorn en segundo plano
port = 8000

# Iniciamos el túnel con ngrok para obtener una URL pública
public_url = ngrok.connect(port).public_url
print("Tunnel URL:", public_url)

# Ejecutamos uvicorn con nuestra app FastAPI
#!kill -9 $(lsof -t -i:8000)  # Para matar procesos en 8000, si ya estaba en uso

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=port)

import threading
server_thread = threading.Thread(target=run_server, args=())
server_thread.start()


ERROR:pyngrok.process.ngrok:t=2025-01-13T15:50:48+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-01-13T15:50:48+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-01-13T15:50:48+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.